# 03 — Classification: predict next-observation direction

This notebook consolidates the classification experiments from the final notebook. It compares Random Forest, SVM-RBF and MLP using both feature sets on validation data. After reviewing those results, it evaluates selected 15-feature models on the held-out test set.

**Important:** test results must not be used to choose a model or tune its settings.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fx_utils import PAIRS, PROCESSED_DIR, RESULTS_DIR, chronological_split, get_7_features

data = {
    pair: pd.read_csv(PROCESSED_DIR / f"{pair}_model_data.csv", index_col=0, parse_dates=True)
    for pair in PAIRS
}
splits = {pair: chronological_split(data[pair]) for pair in PAIRS}

def make_classifier(name):
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=1000, max_depth=100, random_state=42, n_jobs=-1
        )
    if name == "SVM-RBF":
        return make_pipeline(StandardScaler(), SVC(kernel="rbf"))
    if name == "MLP":
        return make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(64, 32), max_iter=500,
                early_stopping=True, random_state=42
            )
        )
    raise ValueError(f"Unknown model: {name}")

In [2]:
# Validation experiments: 3 models × 2 feature sets × 3 currency pairs
validation_rows = []

for pair in PAIRS:
    train, val = splits[pair]["train"], splits[pair]["val"]
    target_col = f"{pair}_target_direction"
    features_7 = get_7_features(pair)
    features_15 = [
        c for c in train.columns
        if c.startswith(f"{pair}_")
        and c not in [f"{pair}_target_return", target_col]
    ]

    for feature_count, feature_cols in [(7, features_7), (15, features_15)]:
        for model_name in ["Random Forest", "SVM-RBF", "MLP"]:
            model = make_classifier(model_name)
            model.fit(train[feature_cols], train[target_col].astype(int))
            predictions = model.predict(val[feature_cols])
            validation_rows.append({
                "Currency Pair": pair,
                "Model": model_name,
                "Features": feature_count,
                "Validation Accuracy": accuracy_score(val[target_col].astype(int), predictions),
                "Balanced Accuracy": balanced_accuracy_score(val[target_col].astype(int), predictions)
            })

validation_results = pd.DataFrame(validation_rows)
display(validation_results.round(4))

,Currency Pair,Model,Features,Validation Accuracy,Balanced Accuracy
0,GBP_USD,Random Forest,7,0.5051,0.5059
1,GBP_USD,SVM-RBF,7,0.5068,0.5119
2,GBP_USD,MLP,7,0.4914,0.4978
3,GBP_USD,Random Forest,15,0.5017,0.5080
4,GBP_USD,SVM-RBF,15,0.4914,0.4960
5,GBP_USD,MLP,15,0.4760,0.4802
6,USD_EUR,Random Forest,7,0.4777,0.4722
7,USD_EUR,SVM-RBF,7,0.5497,0.5099
8,USD_EUR,MLP,7,0.4846,0.4866
9,USD_EUR,Random Forest,15,0.5942,0.5570


In [3]:
# Average validation scores across the three currency pairs
average_validation = (
    validation_results
    .groupby(["Model", "Features"])[["Validation Accuracy", "Balanced Accuracy"]]
    .mean()
    .reset_index()
)
display(average_validation.round(4))

,Model,Features,Validation Accuracy,Balanced Accuracy
0,MLP,7,0.4914,0.4929
1,MLP,15,0.5240,0.5009
2,Random Forest,7,0.4966,0.4942
3,Random Forest,15,0.5274,0.5247
4,SVM-RBF,7,0.5194,0.5043
5,SVM-RBF,15,0.5263,0.4997


In [4]:
# Final test evaluation: 15-feature Random Forest and SVM-RBF vs majority baseline.
# Refit on train + validation after model selection, then evaluate test once.
test_rows = []
test_reports = []
test_prediction_records = []

for pair in PAIRS:
    train, val, test = splits[pair]["train"], splits[pair]["val"], splits[pair]["test"]
    development = pd.concat([train, val])
    target_col = f"{pair}_target_direction"
    feature_cols = [
        c for c in development.columns
        if c.startswith(f"{pair}_")
        and c not in [f"{pair}_target_return", target_col]
    ]

    X_dev, y_dev = development[feature_cols], development[target_col].astype(int)
    X_test, y_test = test[feature_cols], test[target_col].astype(int)

    baseline = DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev)
    baseline_pred = baseline.predict(X_test)
    test_rows.append({
        "Currency Pair": pair, "Model": "Majority baseline", "Features": 0,
        "Test Accuracy": accuracy_score(y_test, baseline_pred),
        "Test Balanced Accuracy": balanced_accuracy_score(y_test, baseline_pred)
    })

    for model_name in ["Random Forest", "SVM-RBF"]:
        model = make_classifier(model_name)
        model.fit(X_dev, y_dev)
        predictions = model.predict(X_test)
        test_rows.append({
            "Currency Pair": pair, "Model": model_name, "Features": len(feature_cols),
            "Test Accuracy": accuracy_score(y_test, predictions),
            "Test Balanced Accuracy": balanced_accuracy_score(y_test, predictions)
        })
        test_reports.append((pair, model_name, classification_report(
            y_test, predictions, zero_division=0
        )))

        # Export out-of-sample predictions for the selected Random Forest model
        if model_name == "Random Forest":
            test_prediction_records.append(pd.DataFrame({
                "date": test.index,
                "pair": pair,
                "y_true": y_test.values,
                "y_pred": predictions
            }))

test_results = pd.DataFrame(test_rows)
display(test_results.round(4))

for pair, model_name, report in test_reports:
    print(f"\n{pair} — {model_name}\n{report}")


,Currency Pair,Model,Features,Test Accuracy,Test Balanced Accuracy
0,GBP_USD,Majority baseline,0,0.5445,0.5000
1,GBP_USD,Random Forest,15,0.5137,0.5267
2,GBP_USD,SVM-RBF,15,0.5017,0.4896
3,USD_EUR,Majority baseline,0,0.5582,0.5000
4,USD_EUR,Random Forest,15,0.5531,0.5633
5,USD_EUR,SVM-RBF,15,0.5548,0.5143
6,USD_JPY,Majority baseline,0,0.5137,0.5000
7,USD_JPY,Random Forest,15,0.4795,0.4801
8,USD_JPY,SVM-RBF,15,0.5223,0.5098



GBP_USD — Random Forest
              precision    recall  f1-score   support

           0       0.58      0.38      0.46       318
           1       0.48      0.67      0.56       266

    accuracy                           0.51       584
   macro avg       0.53      0.53      0.51       584
weighted avg       0.53      0.51      0.50       584


GBP_USD — SVM-RBF
              precision    recall  f1-score   support

           0       0.54      0.63      0.58       318
           1       0.44      0.35      0.39       266

    accuracy                           0.50       584
   macro avg       0.49      0.49      0.49       584
weighted avg       0.49      0.50      0.49       584


USD_EUR — Random Forest
              precision    recall  f1-score   support

           0       0.63      0.48      0.54       326
           1       0.50      0.65      0.56       258

    accuracy                           0.55       584
   macro avg       0.56      0.56      0.55       584
weigh

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
validation_results.to_csv(RESULTS_DIR / "classification_validation_results.csv", index=False)
average_validation.to_csv(RESULTS_DIR / "classification_validation_averages.csv", index=False)
test_results.to_csv(RESULTS_DIR / "classification_test_results.csv", index=False)

# Export combined out-of-sample test predictions for Notebook 07
test_predictions_df = pd.concat(test_prediction_records, ignore_index=True)
test_predictions_path = RESULTS_DIR / "classification_test_predictions.csv"
test_predictions_df.to_csv(test_predictions_path, index=False)

print("Saved classification summary tables to:", RESULTS_DIR)
print("\nSaved test predictions to:", test_predictions_path)
print("  Shape:", test_predictions_df.shape)
print("  Columns:", test_predictions_df.columns.tolist())
print(f"  Date range: {test_predictions_df['date'].min().date()} to {test_predictions_df['date'].max().date()}")
print("\nPrediction count per pair:")
print(test_predictions_df["pair"].value_counts().to_string())


Saved classification summary tables to: /Users/namithar/ML-Projects/FX-Forecasting/results

Saved test predictions to: /Users/namithar/ML-Projects/FX-Forecasting/results/classification_test_predictions.csv
  Shape: (1752, 4)
  Columns: ['date', 'pair', 'y_true', 'y_pred']
  Date range: 2025-03-06 to 2026-10-10

Prediction count per pair:
pair
GBP_USD    584
USD_EUR    584
USD_JPY    584


In [6]:
# Validation cell: Reload saved test predictions and verify schema, integrity, and coverage
reloaded = pd.read_csv(test_predictions_path, parse_dates=["date"])

required_cols = ["date", "pair", "y_true", "y_pred"]
assert list(reloaded.columns) == required_cols, f"Schema mismatch: {reloaded.columns.tolist()} vs {required_cols}"
assert reloaded[required_cols].isna().sum().sum() == 0, f"Found missing values: {reloaded.isna().sum().to_dict()}"
assert not reloaded.duplicated(subset=["date", "pair"]).any(), "Found duplicate (date, pair) rows!"
assert set(reloaded["pair"].unique()) == set(PAIRS), f"Pair coverage mismatch: {reloaded['pair'].unique()}"

for p in PAIRS:
    subset = reloaded[reloaded["pair"] == p]
    assert len(subset) == len(splits[p]["test"]), f"Row count mismatch for {p}: {len(subset)} vs {len(splits[p]['test'])}"
    assert subset["date"].is_monotonic_increasing, f"Dates not sorted chronologically for {p}"

print("="*60)
print("VALIDATION PASSED: classification_test_predictions.csv")
print("="*60)
display(reloaded.groupby("pair").agg(
    total_observations=("date", "count"),
    start_date=("date", lambda s: s.min().date()),
    end_date=("date", lambda s: s.max().date()),
    accuracy=("y_true", lambda y: (y == reloaded.loc[y.index, "y_pred"]).mean())
))


VALIDATION PASSED: classification_test_predictions.csv


,total_observations,start_date,end_date,accuracy
pair,,,,
GBP_USD,584,2025-03-06,2026-10-10,0.513699
USD_EUR,584,2025-03-06,2026-10-10,0.553082
USD_JPY,584,2025-03-06,2026-10-10,0.479452


## How to report results

- Validation results are used for model comparison; test results estimate performance on later unseen observations.
- Compare models with the majority-class baseline and report both accuracy and balanced accuracy.
- Results near 50% indicate limited directional signal. Accuracy alone does not establish trading profitability.
- The data source and time period differ from the original paper, so comparisons are indicative rather than exact replications.